# Séance 4 · Collecter et nettoyer les données · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Aujourd'hui tu vis une journée de data scientist : on prend un vrai jeu de données, on l'abîme exprès (comme dans la vraie vie : trous, doublons, fautes de frappe, dates bizarres), et tu le remets en état, étape par étape. Puis tu vas chercher des données toi-même sur internet grâce à une **API**.

Ce notebook tourne dans **Google Colab** (rien à installer). Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter.

**Livrable de la séance** : un dataset nettoyé, avec un *journal des corrections* qui documente chaque réparation.


## Préparation

Tout est déjà installé dans Colab. On importe juste nos outils.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__, "· prêt !")

## 1. Une journée de data scientist

On imagine souvent le data scientist devant des graphiques magnifiques et des modèles d'IA. La réalité : il passe **la moitié de son temps à nettoyer des données**.

D'où viennent les données ? De partout : formulaires remplis par des humains (avec des fautes), capteurs (qui tombent en panne), applis (dont chaque version enregistre différemment), fichiers Excel copiés-collés... Résultat : un vrai jeu de données est **sale, incomplet et plein de pièges**.

Le métier suit toujours les mêmes 5 étapes :

1. **Collecte** : récupérer les données (fichier, base de données, API)
2. **Nettoyage** : réparer les trous, les doublons, les formats
3. **Analyse** : poser des questions, faire des graphiques (séance 5)
4. **Modélisation** : entraîner un modèle qui prédit (séance 6)
5. **Restitution** : raconter ce qu'on a trouvé à quelqu'un qui décide

Analogie : c'est comme cuisiner. Faire les courses (collecte), laver et éplucher (nettoyage), goûter et ajuster (analyse), suivre la recette (modélisation), servir joliment (restitution). Personne ne sert des légumes pleins de terre.

In [ ]:
# Un petit schéma des 5 étapes, avec le temps que chacune prend dans une vraie journée
etapes = ["Collecte", "Nettoyage", "Analyse", "Modélisation", "Restitution"]
temps = [20, 40, 15, 10, 15]   # en % du temps (ordre de grandeur, ça dépend des projets)

fig, ax = plt.subplots(figsize=(10, 2.8))
gauche = 0
for etape, t in zip(etapes, temps):
    ax.barh(0, t, left=gauche, edgecolor="white", height=0.6)
    ax.text(gauche + t / 2, 0, f"{etape}\n{t} %", ha="center", va="center", color="white", fontsize=10)
    gauche += t
ax.set_xlim(0, 100); ax.axis("off")
ax.set_title("Une journée de data scientist : où passe le temps ?")
plt.show()

## 2. Collecte : charger un jeu de données propre

On reprend les 800 Pokémon de la séance 2. Ce fichier est **propre** : pas de faute, pas de doublon. On le garde de côté comme référence (`pokemon_propre`), pour pouvoir vérifier notre travail à la fin.

`.info()` est le bilan de santé d'un tableau : le type de chaque colonne (Dtype) et le nombre de cases remplies (Non-Null). Regarde `Type 2` : 414 cases remplies sur 800. Ce n'est pas une erreur, beaucoup de Pokémon n'ont qu'un seul type. Une **valeur manquante** (affichée `NaN`, *Not a Number*) n'est pas toujours un bug : il faut comprendre *pourquoi* elle manque.

In [ ]:
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"

try:
    pokemon_propre = pd.read_csv(URL_POKEMON)
    print("Chargé :", pokemon_propre.shape[0], "lignes,", pokemon_propre.shape[1], "colonnes")
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger le fichier :", erreur)

pokemon_propre.head()

In [ ]:
pokemon_propre.info()   # le bilan de santé : types des colonnes et cases remplies

## 3. On l'abîme volontairement

Pour s'entraîner, on va saboter ce tableau comme la vraie vie le ferait. La fonction `abimer` ajoute :

- des **valeurs manquantes** (des cases effacées),
- des **doublons** (des lignes recopiées),
- des **fautes de frappe** dans une catégorie : `fire`, `Fire ` (avec un espace), `Fier`,
- une colonne **date** avec 3 formats mélangés : `2024-01-05`, `05/01/2024`, `5 janv 2024`,
- des **nombres écrits en texte** : `"45 "` au lieu de `45`.

Le `seed` (la graine) fixe le hasard : tout le monde obtient exactement le même tableau abîmé.

In [ ]:
MOIS_FR = ["janv", "fév", "mars", "avr", "mai", "juin", "juil", "août", "sept", "oct", "nov", "déc"]

def abimer(df, col_categorie, col_nombre, seed=42):
    rng = np.random.default_rng(seed)
    sale = df.copy()
    # 1. valeurs manquantes : on efface 5 % des cases de deux colonnes
    for col in [col_categorie, col_nombre]:
        lignes = rng.choice(sale.index, size=len(sale) // 20, replace=False)
        sale.loc[lignes, col] = np.nan
    # 2. fautes de frappe dans la catégorie
    def faute(v):
        if not isinstance(v, str): return v
        r = rng.random()
        if r < 0.04: return v.lower()                       # "fire"
        if r < 0.08: return v + " "                         # "Fire "
        if r < 0.11: return v[:-2] + v[-1] + v[-2]          # "Fier" (deux lettres inversées)
        return v
    sale[col_categorie] = sale[col_categorie].apply(faute)
    # 3. une colonne date avec 3 formats différents
    jours = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, len(sale)), unit="D")
    def ecrire_date(d, style):
        if style == 0: return d.strftime("%Y-%m-%d")             # 2024-01-05
        if style == 1: return d.strftime("%d/%m/%Y")             # 05/01/2024
        return f"{d.day} {MOIS_FR[d.month - 1]} {d.year}"        # 5 janv 2024
    sale["date_capture"] = [ecrire_date(d, s) for d, s in zip(jours, rng.integers(0, 3, len(sale)))]
    # 4. nombres en texte, avec un espace parasite
    sale[col_nombre] = sale[col_nombre].map(lambda v: v if pd.isna(v) else f"{v:g} ")
    # 5. doublons : on recopie 10 lignes au hasard, puis on mélange tout
    sale = pd.concat([sale, sale.sample(10, random_state=seed)])
    return sale.sample(frac=1, random_state=seed).reset_index(drop=True)

df = abimer(pokemon_propre, col_categorie="Type 1", col_nombre="HP")
df.head(8)

**Exercice** : sans lire la fonction `abimer`, trouve les dégâts par toi-même : combien de lignes en trop ? combien de valeurs manquantes par colonne ? Affiche aussi toutes les valeurs différentes de `Type 1` (indice : `df["Type 1"].unique()`).

<details><summary>Solution</summary>

```python
print("Lignes en trop :", len(df) - len(pokemon_propre))
print(df.isna().sum())
print(sorted(df["Type 1"].dropna().unique()))
```
</details>

In [ ]:
# À toi

## 4. Nettoyage : réparer étape par étape

Règle d'or du data scientist : **on note chaque correction**. Si quelqu'un te demande dans 6 mois « pourquoi il n'y a plus que 800 lignes ? », tu dois pouvoir répondre. C'est le rôle du **journal des corrections** : une simple liste Python qu'on remplit au fur et à mesure. À la fin de la séance, c'est ton livrable.

In [ ]:
journal = []   # le journal des corrections : une ligne par réparation

def noter(message):
    journal.append(message)
    print("📝", message)

noter(f"Départ : {len(df)} lignes, {df.isna().sum().sum()} cases vides")

### 4a. Les doublons

Une ligne recopiée deux fois fausse tous les comptages. `duplicated()` repère les lignes identiques à une ligne déjà vue, `drop_duplicates()` les enlève.

In [ ]:
nb_doublons = df.duplicated().sum()
print("Doublons trouvés :", nb_doublons)
print(df[df.duplicated(keep=False)].sort_values("Name").head(4))   # keep=False : montre les deux exemplaires

df = df.drop_duplicates().reset_index(drop=True)
noter(f"Doublons : {nb_doublons} lignes identiques supprimées, il reste {len(df)} lignes")

### 4b. Les fautes de frappe dans les catégories

`fire`, `Fire ` et `Fire` sont trois valeurs différentes pour l'ordinateur. Résultat : le type Feu est compté trois fois séparément. Deux outils :

- `.str.strip()` enlève les espaces au début et à la fin, `.str.capitalize()` met une majuscule et le reste en minuscules,
- pour les lettres inversées (`Fier`), on construit un **mapping** (un dictionnaire « valeur fausse → valeur juste »).

In [ ]:
print("Avant :", df["Type 1"].nunique(), "types différents")
df["Type 1"] = df["Type 1"].str.strip().str.capitalize()
print("Après strip + capitalize :", df["Type 1"].nunique(), "types différents")
df["Type 1"].value_counts().tail(8)   # les valeurs rares en bas de liste sont suspectes

In [ ]:
import difflib   # pour trouver le mot valide le plus proche d'une faute de frappe

types_valides = sorted(pokemon_propre["Type 1"].unique())
suspects = [t for t in df["Type 1"].dropna().unique() if t not in types_valides]

corrections = {}
for faux in suspects:
    proche = difflib.get_close_matches(faux, types_valides, n=1, cutoff=0.6)
    corrections[faux] = proche[0] if proche else faux
print(corrections)

df["Type 1"] = df["Type 1"].replace(corrections)
noter(f"Type 1 : espaces et majuscules normalisés, {len(corrections)} fautes corrigées → {df['Type 1'].nunique()} types valides")

**Exercice** : vérifie que tous les types de `df["Type 1"]` sont maintenant dans `types_valides` (une liste vide de suspects = gagné). Puis affiche le nombre de Pokémon par type avec `value_counts()`.

<details><summary>Solution</summary>

```python
suspects = [t for t in df["Type 1"].dropna().unique() if t not in types_valides]
print("Suspects restants :", suspects)
print(df["Type 1"].value_counts())
```
</details>

In [ ]:
# À toi

### 4c. Les nombres écrits en texte

`"45 "` est un texte, pas un nombre : impossible de calculer une moyenne. On retire l'espace avec `.str.strip()`, puis on **convertit le type** avec `astype(float)`. Pourquoi `float` et pas `int` ? Parce qu'un `NaN` ne peut pas vivre dans une colonne d'entiers.

In [ ]:
print("Type de la colonne HP avant :", df["HP"].dtype)   # object = texte
df["HP"] = df["HP"].str.strip().astype(float)
print("Type de la colonne HP après :", df["HP"].dtype)
noter("HP : texte converti en nombre (strip + astype)")
df["HP"].describe().round(1)

### 4d. Les valeurs manquantes

Trois stratégies, à choisir selon le cas :

- **supprimer** la ligne (`dropna`) : si la case manquante rend la ligne inutile,
- **remplir** avec une valeur raisonnable (`fillna`) : la médiane pour un nombre, la valeur la plus fréquente pour une catégorie,
- **garder** le `NaN` s'il a un sens (comme `Type 2` : « pas de second type »).

Ici on remplit `HP` par la médiane et `Type 1` par `"Inconnu"` (mentir en inventant un type serait pire).

In [ ]:
print(df[["Type 1", "HP"]].isna().sum())

mediane_hp = df["HP"].median()
nb_vides_hp = df["HP"].isna().sum()
df["HP"] = df["HP"].fillna(mediane_hp).astype(int)      # maintenant on peut repasser en entiers
df["Type 1"] = df["Type 1"].fillna("Inconnu")
noter(f"HP : {nb_vides_hp} cases vides remplies par la médiane ({mediane_hp:g})")
noter("Type 1 : cases vides remplacées par 'Inconnu'")

**Exercice** : l'autre choix aurait été de supprimer les lignes sans type. Sur une copie (`test = df.copy()`), remplace les `"Inconnu"` par `np.nan` puis utilise `dropna(subset=["Type 1"])`. Combien de lignes aurait-on perdu ?

<details><summary>Solution</summary>

```python
test = df.copy()
test["Type 1"] = test["Type 1"].replace("Inconnu", np.nan)
test = test.dropna(subset=["Type 1"])
print("Lignes perdues :", len(df) - len(test))
```
</details>

In [ ]:
# À toi

### 4e. Les dates dans tous les formats

`2024-01-05`, `05/01/2024`, `5 janv 2024` : trois façons d'écrire la même chose. Tant que ce sont des textes, impossible de trier par date ou de calculer une durée.

`pd.to_datetime(colonne, format=...)` lit un format précis ; avec `errors="coerce"`, ce qui ne correspond pas devient `NaT` (*Not a Time*) au lieu de planter. Astuce : on essaie chaque format, puis on **recolle** les résultats avec `fillna`.

In [ ]:
texte = df["date_capture"]
essai_iso = pd.to_datetime(texte, format="%Y-%m-%d", errors="coerce")     # 2024-01-05
essai_fr = pd.to_datetime(texte, format="%d/%m/%Y", errors="coerce")      # 05/01/2024
print("Lues au format ISO :", essai_iso.notna().sum(), "· au format jj/mm/aaaa :", essai_fr.notna().sum())

# Le format "5 janv 2024" : on remplace le mois en lettres par son numéro, puis on lit "5 1 2024"
texte_mois = texte.copy()
for numero, mois in enumerate(MOIS_FR, start=1):
    texte_mois = texte_mois.str.replace(mois, str(numero), regex=False)
essai_lettres = pd.to_datetime(texte_mois, format="%d %m %Y", errors="coerce")

df["date_capture"] = essai_iso.fillna(essai_fr).fillna(essai_lettres)
print("Dates non lues :", df["date_capture"].isna().sum())
noter("date_capture : 3 formats convertis en vraies dates (to_datetime)")
df[["Name", "date_capture"]].head()

**Exercice** : maintenant que ce sont de vraies dates, trouve le Pokémon capturé le plus tôt et le plus tard (`min()`, `max()`, `sort_values`), puis compte les captures par mois (indice : `df["date_capture"].dt.month.value_counts().sort_index()`).

<details><summary>Solution</summary>

```python
tri = df.sort_values("date_capture")
print("Première capture :", tri.iloc[0]["Name"], tri.iloc[0]["date_capture"].date())
print("Dernière capture :", tri.iloc[-1]["Name"], tri.iloc[-1]["date_capture"].date())
print(df["date_capture"].dt.month.value_counts().sort_index())
```
</details>

In [ ]:
# À toi

## 5. Vérifier son travail

Un bon nettoyage se prouve. On compare le tableau réparé au tableau propre d'origine : même nombre de lignes ? mêmes types ? mêmes moyennes ? Et on relit le journal.

In [ ]:
print("Lignes   : propre =", len(pokemon_propre), "| réparé =", len(df))
print("Types 1  : propre =", pokemon_propre["Type 1"].nunique(), "| réparé =", df["Type 1"].nunique(), "(dont 'Inconnu')")
print("HP moyen : propre =", round(pokemon_propre["HP"].mean(), 1), "| réparé =", round(df["HP"].mean(), 1))
print()
print("=== Journal des corrections ===")
for i, ligne in enumerate(journal, start=1):
    print(f"{i}. {ligne}")

## 6. Collecte : récupérer des données soi-même avec une API

Une **API** (*Application Programming Interface*) est une porte par laquelle un programme demande des données à un site. Analogie : au restaurant, tu ne vas pas en cuisine, tu passes commande au serveur avec des mots précis, et il revient avec le plat. Ici le serveur, c'est `requests`, la commande, c'est une URL, et le plat arrive en **JSON** (un format qui ressemble à un dictionnaire Python).

Premier essai : la [PokéAPI](https://pokeapi.co), gratuite et sans inscription. Deuxième essai : la météo avec [Open-Meteo](https://open-meteo.com), gratuite et sans clé : on lui donne une latitude et une longitude, elle renvoie la météo actuelle.

In [ ]:
def demander(url, secours):
    # Appelle l'API ; si pas de réseau, renvoie un dict de secours pour que la suite tourne quand même
    try:
        reponse = requests.get(url, timeout=10)
        reponse.raise_for_status()          # erreur si le site répond autre chose que 200 OK
        return reponse.json()
    except Exception as erreur:
        print("Pas de réseau ? On utilise des données de secours.", erreur)
        return secours

SECOURS_PIKACHU = {"name": "pikachu", "height": 4, "weight": 60,
                   "types": [{"type": {"name": "electric"}}],
                   "stats": [{"stat": {"name": "hp"}, "base_stat": 35}, {"stat": {"name": "attack"}, "base_stat": 55},
                             {"stat": {"name": "defense"}, "base_stat": 40}, {"stat": {"name": "speed"}, "base_stat": 90}]}

pikachu = demander("https://pokeapi.co/api/v2/pokemon/pikachu", SECOURS_PIKACHU)
print(type(pikachu), "· clés disponibles :", list(pikachu.keys())[:12], "...")
print(pikachu["name"], "· taille :", pikachu["height"], "· poids :", pikachu["weight"])

In [ ]:
# Le JSON est imbriqué (des dicts dans des listes dans des dicts). On va chercher ce qui nous intéresse.
types_pikachu = [t["type"]["name"] for t in pikachu["types"]]
stats_pikachu = {s["stat"]["name"]: s["base_stat"] for s in pikachu["stats"]}
print("Types :", types_pikachu)
print("Stats :", stats_pikachu)

# Plusieurs Pokémon → un DataFrame. Chaque appel donne une ligne.
def fiche_pokemon(nom):
    data = demander(f"https://pokeapi.co/api/v2/pokemon/{nom}", dict(SECOURS_PIKACHU, name=nom))
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    return {"nom": data["name"], "type": data["types"][0]["type"]["name"],
            "taille": data["height"], "poids": data["weight"], **stats}

equipe = pd.DataFrame([fiche_pokemon(n) for n in ["pikachu", "bulbasaur", "charmander", "squirtle", "snorlax"]])
equipe

In [ ]:
URL_METEO = "https://api.open-meteo.com/v1/forecast?latitude=48.85&longitude=2.35&current_weather=true"
SECOURS_METEO = {"current_weather": {"temperature": 18.0, "windspeed": 12.0, "winddirection": 200, "weathercode": 3, "time": "2024-06-01T12:00"}}

meteo = demander(URL_METEO, SECOURS_METEO)
actuel = meteo["current_weather"]
print("À Paris maintenant :", actuel["temperature"], "°C, vent", actuel["windspeed"], "km/h")
pd.DataFrame([actuel])

**Exercice** : compare la météo de 3 villes en une seule fois. Construis un DataFrame avec une ligne par ville (colonnes `ville`, `temperature`, `vent`). Coordonnées : Paris (48.85, 2.35), Marseille (43.30, 5.37), Lille (50.63, 3.06). Quelle ville est la plus chaude en ce moment ?

<details><summary>Solution</summary>

```python
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}
lignes = []
for ville, (lat, lon) in villes.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    actuel = demander(url, SECOURS_METEO)["current_weather"]
    lignes.append({"ville": ville, "temperature": actuel["temperature"], "vent": actuel["windspeed"]})
meteo_villes = pd.DataFrame(lignes).sort_values("temperature", ascending=False)
print(meteo_villes)
```
</details>

In [ ]:
# À toi
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}

## 7. Projet (80 min) : nettoie ton propre dataset

À toi de jouer. On prend un autre vrai jeu de données : **Tips**, 244 additions d'un restaurant américain (montant, pourboire, jour, nombre de convives...). On l'abîme avec la même fonction, et tu appliques la recette complète **en tenant ton journal**.

Consignes :
1. Charge le dataset propre et garde-le en référence (`tips_propre`).
2. Abîme-le avec `abimer(tips_propre, col_categorie="day", col_nombre="total_bill")`.
3. Répare dans l'ordre : doublons → catégories → nombres → valeurs manquantes → dates. **Une ligne de journal par correction.**
4. Vérifie en comparant au tableau propre (nombre de lignes, moyenne de `total_bill`).
5. Sauvegarde ton fichier propre (`tips_propre.csv`) et affiche le journal : c'est ton livrable.

Bonus : enrichis ta table `equipe` avec 5 autres Pokémon de ton choix via l'API, ou ajoute la météo du jour au tableau Tips.

In [ ]:
# Étape 1 : charger le dataset propre de référence
URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
try:
    tips_propre = pd.read_csv(URL_TIPS)
except Exception as erreur:
    print("Pas de réseau ?", erreur)
tips_propre.head()

In [ ]:
# Étape 2 : abîmer, et faire le diagnostic (info, isna, unique)
tips = abimer(tips_propre, col_categorie="day", col_nombre="total_bill")
journal_tips = []

def noter_tips(message):
    journal_tips.append(message)
    print("📝", message)

noter_tips(f"Départ : {len(tips)} lignes, {tips.isna().sum().sum()} cases vides")
tips.info()

In [ ]:
# Étape 3a : les doublons

# Étape 3b : les catégories (day) → strip, capitalize, mapping des fautes
jours_valides = sorted(tips_propre["day"].unique())

In [ ]:
# Étape 3c : les nombres en texte (total_bill)

# Étape 3d : les valeurs manquantes (day, total_bill) : supprimer ou remplir ? justifie dans le journal

In [ ]:
# Étape 3e : les dates (date_capture)

In [ ]:
# Étape 4 : vérifier (lignes, moyenne de total_bill, info)

In [ ]:
# Étape 5 : sauvegarder et afficher le journal (le livrable)
# tips.to_csv("tips_propre.csv", index=False)   # décommente quand tout est réparé
for i, ligne in enumerate(journal_tips, start=1):
    print(f"{i}. {ligne}")

<details><summary>Solution complète du projet</summary>

```python
# 3a doublons
n = tips.duplicated().sum()
tips = tips.drop_duplicates().reset_index(drop=True)
noter_tips(f"Doublons : {n} lignes supprimées, reste {len(tips)}")

# 3b catégories
tips["day"] = tips["day"].str.strip().str.capitalize()
suspects = [d for d in tips["day"].dropna().unique() if d not in jours_valides]
corrections = {f: difflib.get_close_matches(f, jours_valides, n=1, cutoff=0.5)[0] for f in suspects}
tips["day"] = tips["day"].replace(corrections)
noter_tips(f"day : normalisé, {len(corrections)} fautes corrigées {corrections}")

# 3c nombres
tips["total_bill"] = tips["total_bill"].str.strip().astype(float)
noter_tips("total_bill : texte → nombre")

# 3d manquants : une addition sans montant ne sert à rien → on supprime ; un jour inconnu → 'Inconnu'
n = tips["total_bill"].isna().sum()
tips = tips.dropna(subset=["total_bill"])
tips["day"] = tips["day"].fillna("Inconnu")
noter_tips(f"total_bill : {n} lignes sans montant supprimées ; day vide → 'Inconnu'")

# 3e dates
t = tips["date_capture"]
iso = pd.to_datetime(t, format="%Y-%m-%d", errors="coerce")
fr = pd.to_datetime(t, format="%d/%m/%Y", errors="coerce")
tm = t.copy()
for num, mois in enumerate(MOIS_FR, start=1):
    tm = tm.str.replace(mois, str(num), regex=False)
lettres = pd.to_datetime(tm, format="%d %m %Y", errors="coerce")
tips["date_capture"] = iso.fillna(fr).fillna(lettres)
noter_tips("date_capture : 3 formats → dates")

# 4 vérifier
print(len(tips_propre), len(tips), round(tips_propre["total_bill"].mean(), 2), round(tips["total_bill"].mean(), 2))
tips.to_csv("tips_propre.csv", index=False)
```
</details>

## À retenir

- Un vrai jeu de données est **sale** : le nettoyage, c'est la moitié du métier
- Le bilan de santé : `df.info()`, `df.isna().sum()`, `df.duplicated().sum()`, `df["col"].unique()`
- **Doublons** → `drop_duplicates()` ; **fautes de frappe** → `.str.strip()`, `.str.capitalize()`, un mapping avec `replace`
- **Nombres en texte** → `astype(float)` ; **dates** → `pd.to_datetime(format=..., errors="coerce")`
- **Valeurs manquantes** : supprimer, remplir ou garder, mais toujours **justifier**
- Une **API** = une URL → `requests.get` → du **JSON** → un DataFrame ; toujours un `try/except`
- Chaque correction va dans le **journal** : ton futur toi te dira merci

## Pour montrer aux autres

Pendant les 20 dernières minutes, chacun présente son journal des corrections. Trois questions guides :

1. Quelle réparation a été la plus difficile, et comment l'as-tu trouvée ?
2. Pour les valeurs manquantes, as-tu choisi de supprimer ou de remplir ? Pourquoi ?
3. Quelle donnée irais-tu chercher avec une API pour ton propre projet ?

## Liens gratuits

- PokéAPI, la documentation : https://pokeapi.co/docs/v2
- Open-Meteo, météo sans clé : https://open-meteo.com/en/docs
- Liste d'APIs publiques gratuites : https://github.com/public-apis/public-apis
- pandas, guide « valeurs manquantes » : https://pandas.pydata.org/docs/user_guide/missing_data.html